In [ ]:
import os

# Mapping based on filename prefix
PREFIX_TO_CLASS = {
    "BRAIIM": 0,
    "FRANOC": 2,
    "LIRIBO": 1,
    "TRIAVA": 3,
}

def replace_class_id_in_file(file_path):
    filename = os.path.basename(file_path)
    prefix = filename.split('_')[0]  # everything before first "_", e.g. BRAIIM_001.txt

    # Determine class ID based on prefix
    if prefix not in PREFIX_TO_CLASS:
        print(f"⚠️ Skipping {filename}: unknown prefix '{prefix}'")
        return

    new_class_id = PREFIX_TO_CLASS[prefix]

    # Read and rewrite file
    with open(file_path, 'r') as f:
        lines = f.readlines()

    new_lines = []
    for line in lines:
        parts = line.strip().split()
        if len(parts) != 5:
            continue  # skip malformed lines
        parts[0] = str(new_class_id)
        new_lines.append(" ".join(parts))

    # Overwrite file with new content
    with open(file_path, 'w') as f:
        f.write("\n".join(new_lines) + "\n")

    print(f"✅ Updated {filename} → class {new_class_id}")


def process_folder(root_folder):
    for root, _, files in os.walk(root_folder):
        for file in files:
            if not file.endswith('.txt'):
                continue
            file_path = os.path.join(root, file)
            replace_class_id_in_file(file_path)


folder = "/user/christoph.wald/u15287/big-scratch/02_splitted_data/test_set/test_set_w_new_labels/labels"
process_folder(folder)


import os
import math

def read_labels(file_path):
    """Read YOLO labels and return list of parsed (class_id, cx, cy, w, h) floats."""
    labels = []
    if not os.path.exists(file_path):
        return labels
    with open(file_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 5:
                class_id = int(parts[0])
                values = tuple(float(x) for x in parts[1:])
                labels.append((class_id, *values))
    return labels


def labels_match(l1, l2, tol=1e-4):
    """Check if two YOLO label entries are approximately equal."""
    if l1[0] != l2[0]:
        return False
    return all(math.isclose(a, b, abs_tol=tol) for a, b in zip(l1[1:], l2[1:]))


def compare_label_files(file_a, file_b, tol=1e-4):
    """Compare two label files and return (deleted, added)."""
    labels_a = read_labels(file_a)
    labels_b = read_labels(file_b)

    matched_b = set()
    deleted = 0

    # Count deletions (items in A not matched in B)
    for la in labels_a:
        found = False
        for i, lb in enumerate(labels_b):
            if i not in matched_b and labels_match(la, lb, tol):
                matched_b.add(i)
                found = True
                break
        if not found:
            deleted += 1

    # Count additions (items in B not matched in A)
    added = len(labels_b) - len(matched_b)

    return deleted, added


def compare_label_folders(folder_a, folder_b, tol=1e-4):
    """
    Compare all label files in two folders.
    Returns dict: { "filename.txt": (added, deleted) }
    """
    results = {}

    for root, _, files in os.walk(folder_a):
        for file in files:
            if not file.endswith('.txt'):
                continue

            rel_path = os.path.relpath(os.path.join(root, file), folder_a)
            path_a = os.path.join(folder_a, rel_path)
            path_b = os.path.join(folder_b, rel_path)

            if not os.path.exists(path_b):
                print(f"⚠️ Missing file in new folder: {rel_path}")
                continue

            deleted, added = compare_label_files(path_a, path_b, tol)
            results[rel_path] = (added, deleted)

    return results





In [16]:

# Example usage
old_labels = "/user/christoph.wald/u15287/big-scratch/02_splitted_data/test_set/test_set_w_old_labels/labels"
new_labels = "/user/christoph.wald/u15287/big-scratch/02_splitted_data/test_set/test_set_w_new_labels/labels"

diffs = compare_label_folders(old_labels, new_labels)
print(diffs)


{'LIRIBO_0091.txt': (0, 0), 'FRANOC_0493.txt': (7, 6), 'LIRIBO_0467.txt': (2, 1), 'LIRIBO_0218.txt': (1, 0), 'LIRIBO_0232.txt': (2, 2), 'FRANOC_0513.txt': (5, 2), 'TRIAVA_0187.txt': (0, 0), 'FRANOC_0344.txt': (5, 5), 'LIRIBO_0102.txt': (0, 0), 'FRANOC_0371.txt': (4, 0), 'LIRIBO_0517.txt': (3, 1), 'FRANOC_0672.txt': (1, 0), 'BRAIIM_0170.txt': (32, 9), 'TRIAVA_0501.txt': (12, 44), 'FRANOC_0169.txt': (16, 0), 'LIRIBO_0519.txt': (0, 0), 'LIRIBO_0049.txt': (0, 0), 'TRIAVA_0180.txt': (3, 0), 'LIRIBO_0349.txt': (0, 0), 'FRANOC_0638.txt': (2, 0), 'TRIAVA_0028.txt': (0, 0), 'FRANOC_0535.txt': (0, 0), 'BRAIIM_1161.txt': (0, 0), 'LIRIBO_0640.txt': (0, 0), 'FRANOC_0199.txt': (0, 0), 'TRIAVA_0188.txt': (0, 0), 'LIRIBO_0069.txt': (0, 0), 'BRAIIM_0195.txt': (14, 0), 'FRANOC_0523.txt': (5, 5), 'BRAIIM_1044.txt': (2, 0), 'FRANOC_0471.txt': (1, 0), 'FRANOC_0365.txt': (2, 0), 'LIRIBO_0297.txt': (0, 0), 'FRANOC_0168.txt': (10, 0), 'BRAIIM_0288.txt': (11, 0), 'LIRIBO_0253.txt': (8, 6), 'TRIAVA_0325.txt': (

In [ ]:
from collections import defaultdict

# Example structure of diffs:
# diffs = {
#     "BRAIIM_001.txt": (1, 0),
#     "BRAIIM_002.txt": (0, 0),
#     "FRANOC_003.txt": (3, 1),
#     "TRIAVA_010.txt": (0, 2),
# }

SPECIES_PREFIXES = ["BRAIIM", "FRANOC", "LIRIBO", "TRIAVA"]

def summarize_diffs(diffs):
    summary = defaultdict(lambda: {
        "unchanged_files": 0,
        "changed_files": 0,
        "total_added": 0,
        "total_deleted": 0
    })

    # --- Per-species aggregation ---
    for filename, (added, deleted) in diffs.items():
        prefix = filename.split('_')[0]
        if prefix not in SPECIES_PREFIXES:
            prefix = "UNKNOWN"

        if added == 0 and deleted == 0:
            summary[prefix]["unchanged_files"] += 1
        else:
            summary[prefix]["changed_files"] += 1
            summary[prefix]["total_added labels"] += added
            summary[prefix]["total_deleted labels"] += deleted

    # --- Find top 10 images with most changes ---
    top_changes = sorted(
        diffs.items(),
        key=lambda x: x[1][0] + x[1][1],
        reverse=True
    )[:10]

    return summary, top_changes



summary, top10 = summarize_diffs(diffs)

print("\n📊 Summary per species:")
for species, stats in summary.items():
    print(f"{species}: {stats}")


print("\n🔥 Top 10 files with most total changes:")
for fname, (added, deleted) in top10:
    print(f"{fname}: +{added}, -{deleted} (total {added + deleted})")




📊 Summary per species:
LIRIBO: {'unchanged_files': 88, 'changed_files': 43, 'total_added': 138, 'total_deleted': 56}
FRANOC: {'unchanged_files': 10, 'changed_files': 68, 'total_added': 402, 'total_deleted': 65}
TRIAVA: {'unchanged_files': 20, 'changed_files': 20, 'total_added': 203, 'total_deleted': 110}
BRAIIM: {'unchanged_files': 5, 'changed_files': 35, 'total_added': 548, 'total_deleted': 110}

🔥 Top 10 files with most total changes:
BRAIIM_0720.txt: +53, -43 (total 96)
BRAIIM_0779.txt: +51, -11 (total 62)
TRIAVA_0501.txt: +12, -44 (total 56)
TRIAVA_0590.txt: +47, -1 (total 48)
BRAIIM_0207.txt: +42, -0 (total 42)
BRAIIM_0223.txt: +42, -0 (total 42)
BRAIIM_0170.txt: +32, -9 (total 41)
TRIAVA_0504.txt: +41, -0 (total 41)
BRAIIM_0781.txt: +29, -9 (total 38)
BRAIIM_0796.txt: +23, -12 (total 35)


In [20]:
import os

def sanitize_labels(folder_path):
    """
    Go through all label files in a folder and replace negative numbers with their absolute values.
    """
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            if not file.endswith(".txt"):
                continue  # skip non-label files
            
            file_path = os.path.join(root, file)
            
            with open(file_path, "r") as f:
                lines = f.readlines()
            
            new_lines = []
            for line in lines:
                parts = line.strip().split()
                # Skip empty lines
                if not parts:
                    continue
                
                cls = parts[0]  # first part is class, leave as is
                coords = [str(abs(float(x))) for x in parts[1:]]  # take abs of all coordinates
                new_line = " ".join([cls] + coords)
                new_lines.append(new_line)
            
            # Write back sanitized lines
            with open(file_path, "w") as f:
                f.write("\n".join(new_lines) + "\n")
            
            print(f"Sanitized file: {file_path}")

# Example usage
label_folder = "/user/christoph.wald/u15287/big-scratch/02_splitted_data/test_set/test_set_w_new_labels/labels"
sanitize_labels(label_folder)


Sanitized file: /user/christoph.wald/u15287/big-scratch/02_splitted_data/test_set/test_set_w_new_labels/labels/LIRIBO_0091.txt
Sanitized file: /user/christoph.wald/u15287/big-scratch/02_splitted_data/test_set/test_set_w_new_labels/labels/FRANOC_0493.txt
Sanitized file: /user/christoph.wald/u15287/big-scratch/02_splitted_data/test_set/test_set_w_new_labels/labels/LIRIBO_0467.txt
Sanitized file: /user/christoph.wald/u15287/big-scratch/02_splitted_data/test_set/test_set_w_new_labels/labels/LIRIBO_0218.txt
Sanitized file: /user/christoph.wald/u15287/big-scratch/02_splitted_data/test_set/test_set_w_new_labels/labels/LIRIBO_0232.txt
Sanitized file: /user/christoph.wald/u15287/big-scratch/02_splitted_data/test_set/test_set_w_new_labels/labels/FRANOC_0513.txt
Sanitized file: /user/christoph.wald/u15287/big-scratch/02_splitted_data/test_set/test_set_w_new_labels/labels/TRIAVA_0187.txt
Sanitized file: /user/christoph.wald/u15287/big-scratch/02_splitted_data/test_set/test_set_w_new_labels/labels/

In [21]:
import os
import cv2
import sys
sys.path.append("/user/christoph.wald/u15287/insect_pest_detection/modules")
from modules import draw_box, load_yolo_labels, visualize_yolo_boxes

In [25]:
visualize_yolo_boxes(image_path="/user/christoph.wald/u15287/big-scratch/02_splitted_data/test_set/test_set_w_new_labels_uncropped/images/BRAIIM_0913.jpg",
                      label_path="/user/christoph.wald/u15287/big-scratch/02_splitted_data/test_set/test_set_w_new_labels_uncropped/labels/BRAIIM_0913 copy.txt",
                       output_folder="/user/christoph.wald/u15287/insect_pest_detection/2_1_2_preprocessing")

Saved: /user/christoph.wald/u15287/insect_pest_detection/2_1_2_preprocessing/BRAIIM_0913.jpg
